<a href="https://colab.research.google.com/github/Rayan1156/phishing_email_detection/blob/main/Stress_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip -o distilbert.zip
!ls


Archive:  distilbert.zip
   creating: distilbert/
  inflating: distilbert/model.safetensors  
  inflating: distilbert/training_args.bin  
  inflating: distilbert/tokenizer.json  
  inflating: distilbert/config.json  
  inflating: distilbert/tokenizer_config.json  
distilbert  distilbert.zip  ensemble.pkl  phishing_clean.csv  sample_data


In [2]:
import torch
print("GPU:", torch.cuda.is_available())

GPU: True


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('phishing_clean.csv')
X = df["Email Text"]
y = df["label"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)
print(len(X_train), len(X_val), len(X_test))


14028 1753 1754


In [4]:
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from sklearn.metrics import confusion_matrix

tokenizer = AutoTokenizer.from_pretrained('distilbert')
model = AutoModelForSequenceClassification.from_pretrained('distilbert')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

def predict_proba(texts):
    probs = []
    for i in range(0, len(texts), 32):
        batch = list(texts[i:i + 32])
        enc = tokenizer(batch, truncation=True, max_length=256,
                        padding=True, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**enc).logits.cpu().numpy()
        probs.extend(softmax(logits, axis=1)[:, 1])
    return np.array(probs)

test_proba = predict_proba(list(X_test))
pred = (test_proba >= 0.05).astype(int)
print(confusion_matrix(y_test, pred))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[[1090    8]
 [   9  647]]


In [5]:
X_test_r = X_test.reset_index(drop=True)
y_test_r = y_test.reset_index(drop=True)

fn_idx = [i for i in range(len(y_test_r)) if y_test_r[i] == 1 and pred[i] == 0]
fp_idx = [i for i in range(len(y_test_r)) if y_test_r[i] == 0 and pred[i] == 1]
print("Missed scams:", len(fn_idx), "| False alarms:", len(fp_idx))

for i in fn_idx:
    print("=" * 60)
    print("MISSED SCAM — model risk was:", round(float(test_proba[i]), 3))
    print(X_test_r[i][:500])


Missed scams: 9 | False alarms: 8
MISSED SCAM — model risk was: 0.036
banner life upgraded to a + + effective february 8 , 2002 banner extended conversion privileges on opterm and potomac term . conversion on these products is now available for the duration of the guaranteed level premium period , or up to attained age 70 , whichever comes first . ( this includes the opterm 30 ! ) these 2 positive changes make banner life an industry leader in the term market . if you ' d like to see for yourself just how competitive they are . . . for broker and broker dealer use
MISSED SCAM — model risk was: 0.028
f . r . ee psychic reading special master psychic brianna andrews call for a free reading special ( 503 ) - 752 - 4407 would you like to gain insight on topics like romance , relationships , career , wealth , and what your future has in store ? having trouble making one of life ' s difficult decisions ? wait no longer ! brianna andrews is here ! " she was great , she told me the different a

In [6]:
for i in fp_idx:
    print("=" * 60)
    print("FALSE ALARM — model risk was:", round(float(test_proba[i]), 3))
    print(X_test_r[i][:500])


FALSE ALARM — model risk was: 0.982
re : recovery plan it would be funny if it were not quite so close to the truth !
FALSE ALARM — model risk was: 0.991
...an alternative to the "kind of de-feminized, over-sized, self-centered,
mercenary-minded lady available on the British singles scene,"Glad to see American culture is making it's way into the British bars too :)
God bless us uncivilized bastards, every one.Still, definately something not right about the below. People are now
cheaper then a decent laptop? (ok, so we knew that already)-------------Selling Wedded Bliss Big Business in Thailand
Thu Aug 29,10:19 AM ET
By Andrew MarshallBANGKOK, Tha
FALSE ALARM — model risk was: 0.106
re : boat i checked the boat and it is 17 ft , 7 in . long , it is a capri model # 1750 ch , it has a am / fm cass . the motor is 3 . 0 l mercruiser alpha sterndrive ( 135 hp )
FALSE ALARM — model risk was: 0.999
You are receiving this email because you signed up to 
receive one of our free reports. If you w

In [7]:
citi = X_test_r[fn_idx[7]]
print("Full length (chars):", len(citi))
print(citi)
toks = tokenizer(citi)["input_ids"]
print("Total tokens:", len(toks), "| model only sees first 256")


Full length (chars): 1699
please update your account details with citibank dear citibank customer , we recently noticed one or more attempts to log in to your citibank account from a foreign ip address and we have reasons to believe that there was attempts to compromise it with brute forcing your pin number . no successful login was detected and you have full protection by now . if you recently accessed your account while travelling , the unusual login attempts may have been initiated by you . the login attempt was made from : ip address : 173 . 27 . 187 . 24 isp host : cache - 82 . proxyserver . cis . com by now , we used many techniques to verify the accuracy of the information our users provide us when they register on the site . however , because user verification on the internet is difficult , citibank cannot and does not confirm each user ' s purported identity . thus , we have established an offline verification system to help you evaluate with whom you are dealing with . the sy

In [8]:
val_proba = predict_proba(list(X_val))
X_val_r = X_val.reset_index(drop=True)
y_val_r = y_val.reset_index(drop=True)

caught = [i for i in range(len(y_val_r))
          if y_val_r[i] == 1 and val_proba[i] >= 0.05]
print("Validation scams the champion catches:", len(caught))

import random
random.seed(42)
base_idx = random.sample(caught, 100)
base_texts = [X_val_r[i] for i in base_idx]
print("Base set ready:", len(base_texts))


Validation scams the champion catches: 647
Base set ready: 100


In [9]:
import re

LURE_WORDS = ["verify", "account", "password", "click", "free", "prize",
              "urgent", "confirm", "update", "bank", "winner", "claim",
              "security", "suspended", "login", "billing", "refund"]

LEET = str.maketrans({"a": "4", "e": "3", "i": "1", "o": "0", "s": "5", "t": "7"})

def disguise_leet(text):
    for w in LURE_WORDS:
        text = re.sub(rf"\b{w}\b", w.translate(LEET), text)
    return text

def disguise_spaced(text):
    for w in LURE_WORDS:
        text = re.sub(rf"\b{w}\b", " ".join(w), text)
    return text

BRANDS = {"paypal": "paypa1", "amazon": "amaz0n", "citibank": "cit1bank",
          "ebay": "eb4y", "apple": "app1e", "microsoft": "micros0ft",
          "netflix": "netf1ix", "chase": "ch4se"}

def disguise_domain(text):
    for real, fake in BRANDS.items():
        text = text.replace(real, fake)
    return text

print(disguise_leet(base_texts[0])[:300])
print(disguise_spaced(base_texts[0])[:300])


build your own calbe tv descrambler with only 7 parts ! cable television descrambler - easy to make ! build your own cable television descrambler with only 7 parts from radio shack for under $ 12 . 00 . - - - - - - - - - - - - - - - - - - - - - ( last minute upd473 : " i ' ve seen my letter and inst
build your own calbe tv descrambler with only 7 parts ! cable television descrambler - easy to make ! build your own cable television descrambler with only 7 parts from radio shack for under $ 12 . 00 . - - - - - - - - - - - - - - - - - - - - - ( last minute u p d a t e : " i ' ve seen my letter and


In [10]:
disguises = [("Leetspeak", disguise_leet),
             ("Spaced letters", disguise_spaced),
             ("Look-alike brands", disguise_domain)]

print("CHAMPION (DistilBERT @0.05) — clean recall:",
      round(float((predict_proba(base_texts) >= 0.05).mean()), 3))
for name, fn in disguises:
    p = predict_proba([fn(t) for t in base_texts])
    print(name, "recall:", round(float((p >= 0.05).mean()), 3))


CHAMPION (DistilBERT @0.05) — clean recall: 1.0
Leetspeak recall: 0.99
Spaced letters recall: 1.0
Look-alike brands recall: 1.0


In [11]:
import joblib
ens = joblib.load('ensemble.pkl')

def ens_proba(texts):
    return ens.predict_proba(list(texts))[:, 1]

print("ENSEMBLE (@0.35) — clean recall:",
      round(float((ens_proba(base_texts) >= 0.35).mean()), 3))
for name, fn in disguises:
    p = ens_proba([fn(t) for t in base_texts])
    print(name, "recall:", round(float((p >= 0.35).mean()), 3))


ENSEMBLE (@0.35) — clean recall: 1.0
Leetspeak recall: 1.0
Spaced letters recall: 1.0
Look-alike brands recall: 1.0


In [12]:
for name, fn in disguises:
    changed = sum(1 for t in base_texts if fn(t) != t)
    print(name, "actually changed", changed, "of 100 emails")


Leetspeak actually changed 45 of 100 emails
Spaced letters actually changed 45 of 100 emails
Look-alike brands actually changed 12 of 100 emails


In [13]:
import pandas as pd

local = pd.read_csv('local_test_set.csv')
local['text'] = local['subject'] + ' ' + local['body']
local['y'] = (local['label'] == 'phishing').astype(int)

local['bert'] = predict_proba(list(local['text']))
local['ens'] = ens_proba(list(local['text']))
local['bert_flag'] = (local['bert'] >= 0.05).astype(int)
local['ens_flag'] = (local['ens'] >= 0.35).astype(int)

print(local[['category', 'label', 'bert', 'ens',
             'bert_flag', 'ens_flag']].round(3).to_string())

ph = local[local['y'] == 1]
sf = local[local['y'] == 0]
print("\nPhishing caught — champion:", ph['bert_flag'].sum(), "/", len(ph),
      "| ensemble:", ph['ens_flag'].sum(), "/", len(ph))
print("Safe wrongly flagged — champion:", sf['bert_flag'].sum(), "/", len(sf),
      "| ensemble:", sf['ens_flag'].sum(), "/", len(sf))
print("\nCaught by category (phishing only):")
print(ph.groupby('category')[['bert_flag', 'ens_flag']].sum())


            category     label   bert    ens  bert_flag  ens_flag
0           jazzcash  phishing  0.999  0.885          1         1
1           jazzcash  phishing  0.996  0.901          1         1
2           jazzcash  phishing  0.998  0.891          1         1
3           jazzcash  phishing  0.081  0.926          1         1
4           jazzcash  phishing  0.999  0.959          1         1
5          easypaisa  phishing  0.995  0.875          1         1
6          easypaisa  phishing  0.996  0.884          1         1
7          easypaisa  phishing  1.000  0.916          1         1
8          easypaisa  phishing  0.712  0.903          1         1
9               bank  phishing  0.991  0.799          1         1
10              bank  phishing  0.997  0.891          1         1
11              bank  phishing  0.582  0.882          1         1
12              bank  phishing  0.997  0.923          1         1
13              bank  phishing  0.999  0.700          1         1
14        

In [14]:
leet_texts = [disguise_leet(t.lower()) for t in ph['text']]
p = predict_proba(leet_texts)
print("Champion vs leetspeak local phishing:",
      int((p >= 0.05).sum()), "/", len(leet_texts), "still caught")


Champion vs leetspeak local phishing: 28 / 30 still caught
